<a href="https://colab.research.google.com/github/Maverick-Ansh/seal/blob/main/scratchpad.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SEAL: Self-Adapting Language Models — rebuilt from first principles

**Paper:** Zweiger, Pari, Guo, Akyürek, Kim, Agrawal. *Self-Adapting Language Models.* arXiv:2506.10943v2, NeurIPS 2025. MIT.

---

## The one-sentence idea

A language model is frozen after pretraining. It cannot learn from a new document the way a student learns from a textbook. SEAL asks: **what if the model writes its own study notes, finetunes itself on those notes, and then we use a reinforcement learning loop to teach it to write better notes?**

The notes are called a **self-edit**. The model generates them. They are plain text. We then run gradient descent on that text. The updated model is tested on questions about the original document, *with the document taken away*. If it answers better, the self-edit was good, and we reinforce the act of writing that kind of note.

So there are two nested loops:

```
OUTER LOOP (reinforcement learning, slow)
  teaches the model HOW TO WRITE a good self-edit
  │
  └── INNER LOOP (supervised finetuning, fast)
        applies one self-edit, producing a temporarily updated model
        that updated model is graded, and the grade is the reward
```

The reward is not a human preference and not a regex on an answer. **The reward is the downstream performance of a whole finetuned model.** That is what makes this paper expensive, and interesting.

---

## What we are going to build

Everything from scratch, in this notebook, in the order a person would actually discover it:

| Part | What we build | Why it comes here |
|---|---|---|
| 1 | Hardware, model, and the SQuAD data | Know the substrate before touching it |
| 2 | **The measuring instrument, and its ceiling and floor** | If the ruler is broken, nothing after this means anything |
| 3 | LoRA, hand-written, no `peft` | The inner loop *is* LoRA. We cannot treat it as a box |
| 4 | The inner loop: text in, updated model out, score out | This is `SFT(θ, SE)` from Eq. 1 |
| 5 | Baseline A: finetune on the raw passage | Paper says this barely helps. Check it |
| 6 | Baseline B: the model writes its own notes, no RL yet | Paper says this helps a lot. Check it |
| 7 | The reward, and the ReST-EM outer loop | Eq. 1 → Eq. 3 → Eq. 4, written as code |
| 8 | Run the RL rounds, plot accuracy per round | Figure 4 |
| 9 | Read the self-edits across rounds | Figure 5 |
| 10 | Catastrophic forgetting from sequential edits | Figure 6 |
| 11 | Prompt ablation | Table 10 |
| 12 | Verdict per claim, deviations, what we did not test | The honest part |

---

## The claims, written so they can be proved wrong

This table is the contract. Every number below comes from the paper. Everything we do later is scored against it.

| # | Claim | Paper location | The number that would confirm it | Type |
|---|---|---|---|---|
| **C1** | Finetuning on the raw passage alone barely helps a model answer questions about it | Table 2 | 32.7 → 33.5, a gain of only **+0.8** | mechanism |
| **C2** | Letting the model rewrite the passage into its own synthetic notes, then finetuning on those, helps a lot more | Table 2 | 33.5 → 39.7, a gain of **+6.2** over raw passage | mechanism |
| **C3** | Reinforcement learning on *which notes to write* adds a further large gain | Table 2, Fig. 4 | 39.7 → **47.0** | headline |
| **C4** | After RL, a small self-editing model beats synthetic data written by a much larger model (GPT-4.1) | Table 2 | SEAL **47.0** vs GPT-4.1 synthetic **46.3** | headline |
| **C5** | The gain saturates fast. Two rounds of ReST-EM are enough | Fig. 4 | 39.7 → 43.7 (round 1) → 47.0 (round 2), then flat | mechanism |
| **C6** | Applying self-edits one after another causes catastrophic forgetting of earlier passages | Fig. 6 | accuracy on passage *i* decays as later edits are applied | limitation |
| **C7** | The wording of the self-edit prompt is itself a huge lever, and RL adds roughly +6 to +11 points on top of *every* prompt | Table 10 | implications 39.7→47.0, rewrite 49.4→55.6, no-prompt 13.8→18.9 | mechanism |

Note which ones are load-bearing. **C1 and C2 together are the real thesis**: the raw data is not the best form of the data, and the model can find a better form. C3 says the model can be taught to get even better at finding it.

---

## What this notebook adds that the paper does not do

Two things, and they both come from taking the measurement seriously.

1. **Separating "newly learned" from "already knew."** SQuAD passages come from Wikipedia. A pretrained model already knows a lot of Wikipedia. The paper's base number is 32.7 percent *with no passage at all*, which means roughly a third of these questions are answerable from prior knowledge alone. When accuracy rises to 47.0, how much of that is genuinely new information entering the weights, and how much is the finetuning just sharpening what was already in there? We will split the questions into **already-known** and **not-known** and report the gain separately on each. This is the difference between "the model learned the document" and "the model got better at guessing."

2. **Measuring the grader.** The paper grades answers with GPT-4.1. We have no such grader, so we build a deterministic one. That is a deviation, and deviations in the *measurement* are the dangerous kind. So we will grade every prediction three different ways and check whether the verdict on each claim survives all three. If a claim flips depending on the grader, that fact is the result.


---
# Part 1 — The substrate

Before writing a single line of method, find out exactly what machine we are on and what model will fit on it. Resizing a paper is not about making it smaller. It is about finding the cheapest setup on which the claim can still be proved wrong.

## The paper's compute, and ours

| | Paper | Us |
|---|---|---|
| GPU | 2× H100 or 2× H200 (80 GB each) | 1× Tesla T4 (16 GB), free tier |
| Base model | Qwen2.5-7B | Qwen2.5-0.5B-Instruct |
| One RL round | about 6 hours | target: about 20 minutes |
| Inner loops per round | 750 | target: about 100 |
| Grader | GPT-4.1 through the OpenAI API | deterministic string matching, built in Part 2 |

## Is shrinking the model legitimate?

This is the question that decides whether the whole reproduction is worth anything, so it is worth being careful.

The rule is: **you may shrink anything the paper holds fixed across its own comparison arms. You may not shrink the thing under test.** The thing under test here is the *self-edit generation policy* and the RL loop that trains it. Those stay exactly as the paper specifies. The base model is held fixed across every row of the paper's Table 2, so it is a substrate, not the contribution.

Better still, the paper already ran this experiment itself. Appendix B.7 reports Qwen2.5-3B and Qwen2.5-7B:

| Model | Base, no training | Base model self-edit | SEAL | Ratio of SEAL gain to self-edit gain |
|---|---|---|---|---|
| Qwen2.5-3B | 25.1 | 31.9 | 37.0 | 1.75× |
| Qwen2.5-7B | 32.7 | 39.7 | 47.0 | 2.04× |

So the authors' own data says the effect **shrinks as the model shrinks**. Running at 0.5B extends their table one step further down, and their trend makes a prediction we can check: the ratio should fall below 1.75. If it does, that is a confirmation of their scaling story. If SEAL collapses entirely at 0.5B, that is a boundary on where the method works, which is also a result.

What we must not do is report a collapse at 0.5B as a refutation of a claim the paper made at 7B. That would be dishonest. We will keep the distinction between **refuted** and **out of range** sharp all the way through.


In [2]:
# The basic reason we care: Tesla T4 is compute capability 7.5 (sm_75).
# It has fp16 tensor cores. It has NO native bf16 tensor cores (those start at sm_80, Ampere).
#
# Careful here, this is a real trap:
#   torch.cuda.is_bf16_supported()  returns True on a T4
# because since torch 2.x that function defaults to including_emulation=True.
# The GPU will happily accept a bf16 tensor and then run it through a slow software path.
# You get correct numbers and roughly a 4x slowdown, with no warning at all.
# So we ask the question properly, and we force fp16 for the whole notebook.

import subprocess, sys, os, platform

print("=" * 78)
print("HARDWARE")
print("=" * 78)
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap,driver_version",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())

import torch
print(f"\ntorch             {torch.__version__}")
print(f"cuda available    {torch.cuda.is_available()}")
if torch.cuda.is_available():
    n = torch.cuda.device_count()
    cap = torch.cuda.get_device_capability()
    print(f"gpu count         {n}")
    print(f"compute cap       {cap[0]}.{cap[1]}")
    try:
        native = torch.cuda.is_bf16_supported(including_emulation=False)
    except TypeError:
        native = cap[0] >= 8
    print(f"bf16 (emulated)   {torch.cuda.is_bf16_supported()}   <-- the misleading answer")
    print(f"bf16 (native)     {native}   <-- the true answer, and why we use fp16")
    print(f"fp16 tensorcores  {cap[0] >= 7}   <-- our fast path")
    for i in range(n):
        free, total = torch.cuda.mem_get_info(i)
        print(f"  cuda:{i}  {torch.cuda.get_device_name(i)}  {free/2**30:.2f} GiB free / {total/2**30:.2f} GiB")
print(f"\npython            {platform.python_version()}")
print(f"cpu count         {os.cpu_count()}")

DTYPE = torch.float16          # locked for the whole notebook
DEV   = "cuda:0"
print(f"\nLOCKED: dtype={DTYPE}, primary device={DEV}")

HARDWARE
Tesla T4, 15360 MiB, 7.5, 580.159.04
Tesla T4, 15360 MiB, 7.5, 580.159.04

torch             2.10.0+cu128
cuda available    True
gpu count         2
compute cap       7.5
bf16 (emulated)   True   <-- the misleading answer
bf16 (native)     False   <-- the true answer, and why we use fp16
fp16 tensorcores  True   <-- our fast path
  cuda:0  Tesla T4  14.46 GiB free / 14.56 GiB
  cuda:1  Tesla T4  14.46 GiB free / 14.56 GiB

python            3.12.13
cpu count         4

LOCKED: dtype=torch.float16, primary device=cuda:0


In [3]:
# Check what is already here before installing anything.
# Blind `pip install -U` on Colab is how you get a kernel that needs restarting.
import importlib, subprocess, sys

WANT = ["transformers", "datasets", "accelerate", "numpy", "matplotlib"]
missing = []
for m in WANT:
    try:
        mod = importlib.import_module(m)
        print(f"  present  {m:<16} {getattr(mod, '__version__', '?')}")
    except ImportError:
        print(f"  MISSING  {m}")
        missing.append(m)

if missing:
    print(f"\ninstalling: {missing}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
    print("done, re-import below")
else:
    print("\nnothing to install")

  present  transformers     5.0.0
  present  datasets         4.8.5
  present  accelerate       1.13.0
  present  numpy            2.4.6
  present  matplotlib       3.10.0

nothing to install


## 1.2 — The task, and why SQuAD is the right choice

SEAL's knowledge-incorporation experiment uses SQuAD v1.1. A SQuAD item is a **passage** from Wikipedia plus several **questions** whose answers are literal spans inside that passage.

Normally SQuAD is a reading-comprehension test: you get the passage and the question together, and you find the span. The paper does something different and much harder. It **deletes the passage at test time**. The model sees only the question. The only way to answer is if the content of that passage is sitting in its weights.

That is exactly what makes it a clean test of knowledge incorporation:

- The passage is short enough that the model can fully understand it when it *is* in context. So there is no reading-comprehension confound. Failure means the information did not get into the weights, not that the model could not parse it.
- The questions come with gold answers, so the reward is automatic. No human in the loop.
- Wikipedia passages are self-contained facts, which is what you want to try to inject.

The paper's own words on why this dataset: its passages "can be fully understood in-context by the base model in-context, yet the model cannot reliably answer questions about them *without* that context."

**That sentence is a testable precondition, not a background remark.** It describes a gap between two numbers: high accuracy with the passage, low accuracy without it. If that gap does not exist for our smaller model, then there is no room for SEAL to do anything, and every number we measure afterwards would be noise. Measuring that gap is the entire job of Part 2.


In [4]:
import random, json, collections
from datasets import load_dataset

SEED = 0
random.seed(SEED)

# SQuAD v1.1 validation split. Each row is (title, context, question, answers).
# Several rows share the same `context`. We regroup so that a "passage" is the unit,
# because a passage is what SEAL writes one self-edit about.
raw = load_dataset("rajpurkar/squad", split="validation")
print(f"raw rows: {len(raw)}")

by_ctx = collections.OrderedDict()
for r in raw:
    by_ctx.setdefault(r["context"], {"title": r["title"], "qas": []})
    by_ctx[r["context"]]["qas"].append({
        "question": r["question"],
        # SQuAD gives several human answers per question. Any of them counts as correct.
        "answers": sorted(set(r["answers"]["text"])),
    })

passages = [{"title": v["title"], "context": k, "qas": v["qas"]} for k, v in by_ctx.items()]
print(f"distinct passages: {len(passages)}")

nq = [len(p["qas"]) for p in passages]
print(f"questions per passage: min={min(nq)} median={sorted(nq)[len(nq)//2]} max={max(nq)}")

# Keep passages that are substantial enough to be worth a self-edit, and that carry
# enough questions for a per-passage accuracy to mean anything.
pool = [p for p in passages if len(p["qas"]) >= 4 and 400 <= len(p["context"]) <= 1800]
print(f"pool after filtering (>=4 questions, 400-1800 chars): {len(pool)}")

# Disjoint splits. No passage appears in both, so nothing we RL-train on is ever evaluated.
rng = random.Random(SEED)
rng.shuffle(pool)
EVAL_PASSAGES  = pool[:24]
TRAIN_PASSAGES = pool[24:24 + 64]

print(f"\nEVAL  passages {len(EVAL_PASSAGES):>3}   questions {sum(len(p['qas']) for p in EVAL_PASSAGES):>4}")
print(f"TRAIN passages {len(TRAIN_PASSAGES):>3}   questions {sum(len(p['qas']) for p in TRAIN_PASSAGES):>4}")

ex = EVAL_PASSAGES[0]
print("\n" + "=" * 78)
print(f"EXAMPLE PASSAGE  (title: {ex['title']}, {len(ex['context'])} chars)")
print("=" * 78)
print(ex["context"])
print("-" * 78)
for q in ex["qas"]:
    print(f"  Q: {q['question']}")
    print(f"  A: {q['answers']}")

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

raw rows: 10570
distinct passages: 2067
questions per passage: min=1 median=5 max=30
pool after filtering (>=4 questions, 400-1800 chars): 1721

EVAL  passages  24   questions  138
TRAIN passages  64   questions  349

EXAMPLE PASSAGE  (title: Scottish_Parliament, 1049 chars)
Under the Scotland Act 1998, ordinary general elections for the Scottish Parliament are held on the first Thursday in May every four years (1999, 2003, 2007 and so on). The date of the poll may be varied by up to one month either way by the Monarch on the proposal of the Presiding Officer. If the Parliament itself resolves that it should be dissolved (with at least two-thirds of the Members voting in favour), or if the Parliament fails to nominate one of its members to be First Minister within 28 days of a General Election or of the position becoming vacant, the Presiding Officer proposes a date for an extraordinary general election and the Parliament is dissolved by the Queen by royal proclamation. Extraordinary g

In [5]:
import torch, time
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"   # one flag. swap to 1.5B / 3B if the budget allows.

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
# Left padding is required for batched *generation*: with right padding the pad tokens
# sit between the prompt and the first generated token and the model attends to garbage.
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,          # transformers v5 renamed torch_dtype -> dtype
    attn_implementation="sdpa",
).to(DEV)
model.eval()
print(f"loaded in {time.time()-t0:.1f}s")

n_params = sum(p.numel() for p in model.parameters())
print(f"parameters       {n_params/1e6:.1f} M")
print(f"weight memory    {n_params*2/2**30:.2f} GiB (fp16)")
print(f"gpu allocated    {torch.cuda.memory_allocated(DEV)/2**30:.2f} GiB")
print(f"hidden size      {model.config.hidden_size}")
print(f"layers           {model.config.num_hidden_layers}")
print(f"vocab            {model.config.vocab_size}")

# Smallest possible proof that the thing works at all.
msgs = [{"role": "user", "content": "Answer in one word. What is the capital of France?"}]
prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
ids = tok(prompt, return_tensors="pt").to(DEV)
with torch.no_grad():
    out = model.generate(**ids, max_new_tokens=8, do_sample=False,
                         pad_token_id=tok.pad_token_id)
print("\nsanity generation:", repr(tok.decode(out[0, ids['input_ids'].shape[1]:], skip_special_tokens=True)))

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


loaded in 11.3s
parameters       494.0 M
weight memory    0.92 GiB (fp16)
gpu allocated    0.93 GiB
hidden size      896
layers           24
vocab            151936

sanity generation: 'Paris'


---
# Part 2 — Build the ruler, then check the ruler

The single most common way a reproduction dies is that the evaluation is subtly broken and everybody spends GPU hours optimizing against a bent ruler. So we build the measurement first, and we refuse to run the expensive part until the measurement has passed three checks.

## The three things we must know before spending any compute

**1. The ceiling.** How well does the model answer these questions when the passage *is* in context? This is the best any amount of weight-editing could hope for, because it is what "perfectly knowing the passage" looks like. If the ceiling is low, the model cannot read, and no self-edit will save it.

**2. The floor, and the shortcut.** How well does it do with no passage and no training? The paper reports 32.7 for Qwen2.5-7B. That number is not zero, and it is worth pausing on why. These are Wikipedia questions, and a pretrained model has read Wikipedia. So part of the floor is genuine prior knowledge, and part is a **guessing shortcut**: "What day of the week are elections held?" has seven possible answers and a model that always says "Thursday" will be right surprisingly often.

  This matters enormously for interpreting the headline claim. If base is 32.7 and SEAL is 47.0, we want to know: is that 14-point gain *new information entering the weights*, or is it the finetuning sharpening answers the model could already produce? The paper does not separate these. We will, by splitting the questions into the ones the base model already gets right and the ones it does not.

**3. The grader's own error.** The paper grades with GPT-4.1, a language model reading the gold answer and the prediction and saying yes or no. We do not have that. We will write a deterministic grader. Any deterministic grader is stricter and dumber than GPT-4.1 in specific ways: it will mark "the Monarch" against gold "Monarch" however we choose to handle articles, it will not know that "28" and "28 days" mean the same thing unless we tell it.

  Rather than pick one and hope, we grade every prediction **three ways at once** and carry all three through the whole notebook:

  - **EM**: exact match after normalization (lowercase, strip articles, strip punctuation). The official SQuAD metric. Strictest.
  - **Contains**: the normalized gold answer appears somewhere inside the normalized prediction. This is the closest cheap approximation to what GPT-4.1 was told to do, since its grading prompt says a student answer "can include additional information, but it must at least fully convey the gold answer."
  - **F1 ≥ 0.6**: token overlap between prediction and gold, thresholded. Catches partial credit that the other two miss.

  If a claim comes out the same under all three, the grader is not driving the result. If the claim flips, the grader is the result, and we say so.

The rule we hold ourselves to: **"the instrument could not measure this" is never reported as "the paper is wrong."**


In [6]:
import re, string, collections
from typing import List, Dict, Optional

# ----------------------------------------------------------------------------------
# SQuAD answer normalisation, written out rather than imported, because every
# judgement call in here changes the numbers and we want them visible.
# This is the official SQuAD v1.1 recipe (Rajpurkar et al. 2016).
# ----------------------------------------------------------------------------------
def normalize(s: str) -> str:
    s = s.lower()
    s = "".join(ch for ch in s if ch not in set(string.punctuation))   # drop punctuation
    s = re.sub(r"\b(a|an|the)\b", " ", s)                              # drop articles
    return " ".join(s.split())                                         # collapse whitespace

def _f1(pred: str, gold: str) -> float:
    p, g = normalize(pred).split(), normalize(gold).split()
    if not p or not g:
        return float(p == g)
    common = collections.Counter(p) & collections.Counter(g)
    n = sum(common.values())
    if n == 0:
        return 0.0
    prec, rec = n / len(p), n / len(g)
    return 2 * prec * rec / (prec + rec)

# The three graders. Each takes a prediction and the list of acceptable gold answers
# (SQuAD gives several human-written answers, any one of which counts).
def grade_em(pred, golds):       return float(any(normalize(pred) == normalize(g) for g in golds))
def grade_contains(pred, golds): return float(any(normalize(g) in normalize(pred) and normalize(g) != ""
                                                  for g in golds))
def grade_f1(pred, golds):       return float(max(_f1(pred, g) for g in golds) >= 0.6)

GRADERS = {"EM": grade_em, "Contains": grade_contains, "F1>=0.6": grade_f1}

# ----------------------------------------------------------------------------------
# The answering prompt. Taken verbatim from the paper, Appendix B.4:
#     "Let's answer a question directly and concisely.
#      Question: {question}
#      Answer:"
# Deviation: the paper uses Qwen2.5-7B *base* with a raw completion prompt. We use an
# Instruct model, so we place the same words inside its chat template. Mixing a base-model
# prompt with an instruct-tuned model is a known way to get garbage, so we adapt the
# wrapper while keeping the paper's wording intact.
# ----------------------------------------------------------------------------------
QA_INSTRUCTION = "Let's answer a question directly and concisely."

def qa_prompt(question: str, context: Optional[str] = None) -> str:
    body = QA_INSTRUCTION
    if context is not None:
        body += f"\n\nPassage:\n{context}"
    body += f"\n\nQuestion: {question}\nAnswer:"
    return tok.apply_chat_template([{"role": "user", "content": body}],
                                   tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def generate(m, prompts: List[str], max_new_tokens=24, temperature=0.0, batch_size=32) -> List[str]:
    """Batched generation. temperature=0 means greedy, which is what evaluation must use:
    a stochastic evaluator adds variance that looks exactly like a real effect."""
    m.eval()
    outs = []
    for i in range(0, len(prompts), batch_size):
        chunk = prompts[i:i + batch_size]
        enc = tok(chunk, return_tensors="pt", padding=True, truncation=True, max_length=1024).to(DEV)
        gen = m.generate(
            **enc, max_new_tokens=max_new_tokens,
            do_sample=temperature > 0,
            **({"temperature": temperature, "top_p": 0.95} if temperature > 0 else {}),
            pad_token_id=tok.pad_token_id,
        )
        for j in range(len(chunk)):
            outs.append(tok.decode(gen[j, enc["input_ids"].shape[1]:], skip_special_tokens=True).strip())
    return outs

def first_line(s: str) -> str:
    """Models like to keep talking. The answer is the first line."""
    return s.split("\n")[0].strip().strip('."')

def evaluate(m, passages, with_context: bool, max_new_tokens=24, batch_size=32) -> Dict:
    """Returns per-grader accuracy plus the per-question records, which we need later
    to split 'already knew' from 'newly learned'."""
    prompts, meta = [], []
    for p in passages:
        for qa in p["qas"]:
            prompts.append(qa_prompt(qa["question"], p["context"] if with_context else None))
            meta.append((p["title"], qa["question"], qa["answers"]))
    preds = [first_line(x) for x in generate(m, prompts, max_new_tokens, 0.0, batch_size)]

    recs = []
    for (title, q, golds), pred in zip(meta, preds):
        recs.append({"title": title, "question": q, "golds": golds, "pred": pred,
                     **{k: g(pred, golds) for k, g in GRADERS.items()}})
    acc = {k: 100.0 * sum(r[k] for r in recs) / len(recs) for k in GRADERS}
    return {"acc": acc, "records": recs, "n": len(recs)}

print("evaluation machinery defined")
print("graders:", list(GRADERS))
print("\nexample prompt (no context):")
print(qa_prompt("What day of the week are general elections held?"))

evaluation machinery defined
graders: ['EM', 'Contains', 'F1>=0.6']

example prompt (no context):
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Let's answer a question directly and concisely.

Question: What day of the week are general elections held?
Answer:<|im_end|>
<|im_start|>assistant



In [7]:
import time
t0 = time.time()

# FLOOR: no passage, no adaptation. This is the paper's "Base model" row (32.7 for 7B).
floor   = evaluate(model, EVAL_PASSAGES, with_context=False)
# CEILING: passage in context. The paper's implicit assumption that these passages
# "can be fully understood in-context". Nobody reports this number. It is the most
# important number in the whole setup.
ceiling = evaluate(model, EVAL_PASSAGES, with_context=True)

print(f"evaluated {floor['n']} questions x2 in {time.time()-t0:.1f}s\n")
print(f"{'grader':<10} {'FLOOR (no passage)':>20} {'CEILING (passage in ctx)':>26} {'GAP':>8}")
print("-" * 68)
for g in GRADERS:
    lo, hi = floor["acc"][g], ceiling["acc"][g]
    print(f"{g:<10} {lo:>19.1f}% {hi:>25.1f}% {hi-lo:>7.1f}")

print(f"\npaper, Qwen2.5-7B: floor 32.7%, ceiling not reported, SEAL reaches 47.0%")

# ---- GATE -------------------------------------------------------------------------
print("\n" + "=" * 68)
gap = ceiling["acc"]["Contains"] - floor["acc"]["Contains"]
if gap < 15:
    print(f"VERDICT: FAIL. Dynamic range is only {gap:.1f} points.")
    print("The model cannot use the passage even when it is handed to it.")
    print("Nothing downstream would be measurable. Stop and fix the setup.")
else:
    print(f"VERDICT: PASS. Dynamic range is {gap:.1f} points (Contains).")
    print(f"SEAL has room to move the number between {floor['acc']['Contains']:.1f}% and {ceiling['acc']['Contains']:.1f}%.")
print("=" * 68)

evaluated 138 questions x2 in 10.3s

grader       FLOOR (no passage)   CEILING (passage in ctx)      GAP
--------------------------------------------------------------------
EM                         0.0%                      11.6%    11.6
Contains                   6.5%                      70.3%    63.8
F1>=0.6                    0.0%                      20.3%    20.3

paper, Qwen2.5-7B: floor 32.7%, ceiling not reported, SEAL reaches 47.0%

VERDICT: PASS. Dynamic range is 63.8 points (Contains).
SEAL has room to move the number between 6.5% and 70.3%.


In [8]:
print("=" * 78)
print("WHY DO THE GRADERS DISAGREE SO MUCH?  (ceiling run, passage in context)")
print("=" * 78)
for r in ceiling["records"][:8]:
    print(f"Q     {r['question']}")
    print(f"gold  {r['golds']}")
    print(f"pred  {r['pred']!r}")
    print(f"      EM={r['EM']:.0f}  Contains={r['Contains']:.0f}  F1>=0.6={r['F1>=0.6']:.0f}")
    print()

import numpy as np
lens = [len(normalize(r["pred"]).split()) for r in ceiling["records"]]
glens = [min(len(normalize(g).split()) for g in r["golds"]) for r in ceiling["records"]]
print(f"prediction length, words:  median {int(np.median(lens))}, mean {np.mean(lens):.1f}")
print(f"gold length, words:        median {int(np.median(glens))}, mean {np.mean(glens):.1f}")

WHY DO THE GRADERS DISAGREE SO MUCH?  (ceiling run, passage in context)
Q     What day of the week are general elections held?
gold  ['Thursday']
pred  'The passage states that "Ordinary general elections for the Scottish Parliament are held on the first Thursday in May every four years'
      EM=0  Contains=1  F1>=0.6=0

Q     What month, every four years, are the ordinary general elections held on?
gold  ['May']
pred  'The ordinary general elections are held on the first Thursday in May every four years'
      EM=0  Contains=1  F1>=0.6=0

Q     Who may change the date by up to a month, on the proposal of the PO?
gold  ['Monarch', 'the Monarch']
pred  'The Presiding Officer may propose a date for an extraordinary general election by up to one month from the proposed date of the'
      EM=0  Contains=0  F1>=0.6=0

Q     If an extraordinary election is held within less than six months before the date of an ordinary election, what does it do to the ordinary election?
gold  ['reverts to t

### The first eval bug, found before spending any GPU time

The diagnostic above is unambiguous. With the passage in context the model *knows the answers*. Look at row 6: gold is "Dorotheenstadt and Friedrichstadt" and the model says "The two Huguenot neighborhoods created in Berlin were Dorotheenstadt and Friedrichstadt." That is a completely correct answer. EM scores it **0**. F1 scores it **0**, because 11 words of prediction against 2 words of gold gives a precision of about 0.18 and no threshold can rescue that.

So two of our three graders are measuring **verbosity**, not correctness.

Why did this not happen to the authors? Because they used `Qwen2.5-7B`, the **base** model, with a raw completion prompt ending in `Answer:`. A base model continues the pattern and emits a short span. We are using an Instruct model, which has been tuned to answer in polite full sentences. The model changed, so the instrument broke. This is the standard way a resized reproduction goes wrong.

There are two honest repairs and we apply both.

**Repair 1: ask for the answer, not an essay.** Add a terseness instruction so the Instruct model behaves like the base model the metric was designed for. This restores the paper's intent rather than departing from it, and we verify it by checking that the ceiling rises for EM and F1 while staying put for Contains. If the ceiling for Contains does not move, then no *new* correctness appeared and all we did was remove padding words.

**Repair 2: find out how often `Contains` lies.** `Contains` is generous by construction. A rambling answer that happens to include the word "May" gets full credit on a question whose gold is "May". So we run a control: grade every prediction against a **randomly reassigned gold answer from a different question**. A perfect grader would score near zero on that. Whatever it does score is the grader's false-positive floor, and every real number has to be read against it.


In [9]:
# ---- Repair 1: try answer-prompt variants, measure the bracket each one gives -------
VARIANTS = {
    "paper-verbatim": "Let's answer a question directly and concisely.",
    "terse":          "Let's answer a question directly and concisely. Reply with the answer only, in as few words as possible. Do not write a sentence.",
    "terse+span":     "Answer the question with the exact short phrase that answers it. No sentence, no explanation, just the phrase.",
}

def qa_prompt_v(question, context, instruction):
    body = instruction
    if context is not None:
        body += f"\n\nPassage:\n{context}"
    body += f"\n\nQuestion: {question}\nAnswer:"
    return tok.apply_chat_template([{"role": "user", "content": body}],
                                   tokenize=False, add_generation_prompt=True)

def eval_with(instruction, passages, with_context, m=None):
    m = m or model
    prompts, meta = [], []
    for p in passages:
        for qa in p["qas"]:
            prompts.append(qa_prompt_v(qa["question"], p["context"] if with_context else None, instruction))
            meta.append(qa["answers"])
    preds = [first_line(x) for x in generate(m, prompts, 24, 0.0, 32)]
    acc = {k: 100.0 * sum(g(pr, gl) for pr, gl in zip(preds, meta)) / len(preds) for k, g in GRADERS.items()}
    wlen = float(np.mean([len(normalize(p).split()) for p in preds]))
    return acc, wlen, preds

print(f"{'variant':<16} {'ctx':<6} {'EM':>7} {'Contains':>10} {'F1>=0.6':>9} {'pred words':>11}")
print("-" * 66)
bracket = {}
for name, instr in VARIANTS.items():
    for wc in (False, True):
        acc, wlen, _ = eval_with(instr, EVAL_PASSAGES, wc)
        bracket[(name, wc)] = acc
        print(f"{name:<16} {str(wc):<6} {acc['EM']:>6.1f}% {acc['Contains']:>9.1f}% {acc['F1>=0.6']:>8.1f}% {wlen:>11.1f}")

print()
print("DYNAMIC RANGE  (ceiling minus floor, bigger is a better instrument)")
print(f"{'variant':<16} {'EM':>10} {'Contains':>16} {'F1>=0.6':>10}")
print("-" * 56)
for name in VARIANTS:
    lo, hi = bracket[(name, False)], bracket[(name, True)]
    print(f"{name:<16} {hi['EM']-lo['EM']:>9.1f} {hi['Contains']-lo['Contains']:>15.1f} {hi['F1>=0.6']-lo['F1>=0.6']:>9.1f}")

variant          ctx         EM   Contains   F1>=0.6  pred words
------------------------------------------------------------------
paper-verbatim   False     0.0%       6.5%      0.0%        13.0
paper-verbatim   True     11.6%      70.3%     20.3%        10.7
terse            False     0.7%       5.8%      2.2%         9.1
terse            True     39.9%      68.8%     50.0%         6.0
terse+span       False     0.7%       3.6%      2.9%         6.7
terse+span       True     51.4%      68.8%     59.4%         4.4

DYNAMIC RANGE  (ceiling minus floor, bigger is a better instrument)
variant                  EM         Contains    F1>=0.6
--------------------------------------------------------
paper-verbatim        11.6            63.8      20.3
terse                 39.1            63.0      47.8
terse+span            50.7            65.2      56.5


In [10]:
# Lock in the repaired instrument for the rest of the notebook.
QA_INSTRUCTION = VARIANTS["terse+span"]

def qa_prompt(question, context=None):
    return qa_prompt_v(question, context, QA_INSTRUCTION)

floor   = evaluate(model, EVAL_PASSAGES, with_context=False)
ceiling = evaluate(model, EVAL_PASSAGES, with_context=True)

# ---- Repair 2: the shuffled-gold control -------------------------------------------
# Grade every prediction against a gold answer belonging to a DIFFERENT question.
# A grader that only rewards correctness scores ~0 here. Whatever it scores is the
# rate at which it hands out credit for nothing, and it is the true zero of the scale.
def shuffled_control(records, n_rep=20, seed=0):
    rng = random.Random(seed)
    out = {k: [] for k in GRADERS}
    idx = list(range(len(records)))
    for _ in range(n_rep):
        perm = idx[:]
        rng.shuffle(perm)
        # make sure nothing is paired with its own gold
        perm = [p if p != i else (p + 1) % len(idx) for i, p in enumerate(perm)]
        for k, g in GRADERS.items():
            out[k].append(100.0 * sum(g(records[i]["pred"], records[j]["golds"])
                                      for i, j in zip(idx, perm)) / len(idx))
    return {k: (float(np.mean(v)), float(np.std(v))) for k, v in out.items()}

ctrl_floor   = shuffled_control(floor["records"])
ctrl_ceiling = shuffled_control(ceiling["records"])

print("=" * 86)
print("THE REPAIRED INSTRUMENT, WITH ITS FALSE-POSITIVE FLOOR")
print("=" * 86)
print(f"{'grader':<10} {'no-passage':>12} {'in-context':>12} {'range':>8}  {'|':<2} {'random-gold control':>22}")
print("-" * 86)
for g in GRADERS:
    lo, hi = floor["acc"][g], ceiling["acc"][g]
    cm, cs = ctrl_ceiling[g]
    print(f"{g:<10} {lo:>11.1f}% {hi:>11.1f}% {hi-lo:>7.1f}  |  {cm:>8.1f}% +/- {cs:.1f}")

print()
print("Reading this table:")
for g in GRADERS:
    cm, _ = ctrl_ceiling[g]
    usable = ceiling["acc"][g] - max(floor["acc"][g], cm)
    verdict = "USABLE" if usable > 20 else "TOO NARROW"
    print(f"  {g:<10} true usable range = {usable:>5.1f} points   -> {verdict}")

THE REPAIRED INSTRUMENT, WITH ITS FALSE-POSITIVE FLOOR
grader       no-passage   in-context    range  |     random-gold control
--------------------------------------------------------------------------------------
EM                 0.7%        51.4%    50.7  |       0.2% +/- 0.3
Contains           3.6%        68.8%    65.2  |       0.2% +/- 0.3
F1>=0.6            2.9%        59.4%    56.5  |       0.2% +/- 0.3

Reading this table:
  EM         true usable range =  50.7 points   -> USABLE
  Contains   true usable range =  65.2 points   -> USABLE
  F1>=0.6    true usable range =  56.5 points   -> USABLE


In [11]:
PRIMARY = "Contains"   # closest to the paper's GPT-4.1 rubric; EM and F1 reported alongside

# Which eval questions does the base model ALREADY answer with no passage at all?
# At 7B the paper's floor is 32.7%, meaning a third of its headline gain could in
# principle be prior knowledge being sharpened rather than new knowledge arriving.
# We check how big that confound is for us.
already_known = {(r["title"], r["question"]) for r in floor["records"] if r[PRIMARY] > 0}
n_total = len(floor["records"])
print(f"eval questions total                  {n_total}")
print(f"answerable with NO passage (base)     {len(already_known)}  ({100*len(already_known)/n_total:.1f}%)")
print(f"genuinely unknown to the model        {n_total-len(already_known)}  ({100*(n_total-len(already_known))/n_total:.1f}%)")
print()
print("Questions the model already knows without reading anything:")
for r in floor["records"]:
    if r[PRIMARY] > 0:
        print(f"  [{r['title']}] {r['question']}")
        print(f"      gold={r['golds']}  pred={r['pred']!r}")

print()
print(f"paper floor (Qwen2.5-7B):  32.7%  ->  about a third of its questions are prior knowledge")
print(f"our floor  (Qwen2.5-0.5B): {floor['acc'][PRIMARY]:.1f}%  ->  the confound is almost absent here")

eval questions total                  138
answerable with NO passage (base)     5  (3.6%)
genuinely unknown to the model        133  (96.4%)

Questions the model already knows without reading anything:
  [American_Broadcasting_Company] What was Walt Disney's brother's name?
      gold=['Roy']  pred="Walt Disney's brother's name was Roy Disney"
  [Warsaw] What pope as a native of Poland?
      gold=['John Paul II']  pred='Pope John Paul II was a native of Poland'
  [Victoria_and_Albert_Museum] Which sculpture by Michelangelo has a full-size replica in the Cast Courts?
      gold=['David', 'David.']  pred='The sculpture by Michelangelo that has a full-size replica in the Castles is "David'
  [Immune_system] What co-receptor recruits molecules inside the T cell that are responsible for cell activation?
      gold=['CD4', 'CD4 co-receptor']  pred='CD4 Co-receptors'
  [Immune_system] Activation of a helper T cell causes it to release what chemicals that influence cell activity?
      gold=[

### Part 2 verdict: the ruler is straight, and it told us something about the paper

Where we ended up:

| | no passage | passage in context | usable range | credit-for-nothing |
|---|---|---|---|---|
| EM | 0.7% | 51.4% | 50.7 | 0.2% |
| **Contains** | **3.6%** | **68.8%** | **65.2** | **0.2%** |
| F1 ≥ 0.6 | 2.9% | 59.4% | 56.5 | 0.2% |

All three graders now have real range and essentially no false-positive rate. We take **Contains** as primary because it matches what the paper actually instructed GPT-4.1 to do, and we carry EM and F1 alongside so no conclusion can rest on one grader.

Two things worth keeping in mind for the rest of the notebook.

**The instrument nearly destroyed the project, and it had nothing to do with the method.** The first measurement said the model scored 11.6% even with the passage sitting in front of it. That looks like a model too small to do the task. It was actually a model that writes in complete sentences being scored by a metric that demands a bare noun phrase. Had we skipped this part and gone straight to training, we would have run the entire RL loop inside an 11-point window, watched nothing move, and concluded the paper does not replicate at small scale. The honest lesson is that on a resized reproduction the evaluation breaks more often than the model does.

**Our floor is 3.6% where the paper's is 32.7%, and that is a real difference in what is being measured.** Those five questions the 0.5B model gets without any passage are ordinary world knowledge: Roy Disney, John Paul II, Michelangelo's David, CD4, cytokines. Nothing else in the eval set is reachable from priors.

This cuts in a direction that helps us. At 7B, roughly a third of the paper's questions are answerable before any adaptation happens, so when the number moves from 32.7 to 47.0 it is genuinely unclear how much of that is new information entering the weights and how much is finetuning sharpening what the model already half-knew. The paper does not separate these. At 0.5B there is almost nothing to sharpen, so **any gain we measure is close to pure knowledge incorporation.** Our absolute numbers will be far below the paper's. Our numbers are, in this one narrow sense, cleaner.

Now we can build the method.


---
# Part 3 — LoRA from scratch

SEAL's inner loop finetunes the model on a piece of text the model just wrote, measures the result, and then **throws the update away**. That happens hundreds of times. So the update mechanism has to be cheap to create, cheap to train, and cheap to delete. That is exactly what LoRA is for, and it is about thirty lines.

## The idea

A weight matrix `W` in the network has shape `(out, in)`. Finetuning normally changes every entry of it. LoRA says: do not touch `W` at all. Instead learn a **correction** `ΔW` and use `W + ΔW`. And force that correction to be low rank, by writing it as a product of two thin matrices:

$$\Delta W = \frac{\alpha}{r}\, B A, \qquad A \in \mathbb{R}^{r \times \text{in}}, \quad B \in \mathbb{R}^{\text{out} \times r}$$

with `r` much smaller than either dimension. In our model `in = out = 896` and we use `r = 32`, so instead of 802,816 numbers per matrix we learn 2 × 32 × 896 = 57,344. About fourteen times fewer.

In the forward pass we never actually build `ΔW`. We route the input down two paths and add:

$$y = Wx + \frac{\alpha}{r} B(Ax)$$

The right-hand path costs `r·in + out·r` multiplies instead of `out·in`, so it is cheap in time as well as memory.

## The two details that matter and are easy to get wrong

**`B` starts at exactly zero.** Then `ΔW = BA = 0` at step 0, so the adapted model is *bit-identical* to the base model before any training. This is what makes "attach a fresh adapter" a safe operation. If both `A` and `B` were random, attaching an adapter would immediately damage the model and every measurement would be polluted by that damage. We will assert this.

**The `α/r` scaling exists so that changing `r` does not change the effective learning rate.** Without it, doubling the rank roughly doubles the size of the update for the same optimizer settings, and every rank sweep would secretly be a learning-rate sweep. The paper uses `r = 32, α = 64`, so our scaling is 2.

## Why we write it instead of importing `peft`

Three reasons, and they are practical rather than ideological.

1. The inner loop needs `attach → train → measure → reset to zero → attach again`, hundreds of times, without reloading a 1 GB model from disk each time. That is not what a library wrapper is shaped for, and fighting it costs more than writing it.
2. On a T4 we must control precision by hand. Base weights stay in fp16 for the tensor cores. The adapter parameters stay in fp32, because fp16 gradients on parameters this small underflow to zero and you get silent no-op training.
3. If the adapter is a box, then the inner loop is a box, and the inner loop is the paper.


In [16]:
import math
import torch.nn as nn

class LoRALinear(nn.Module):
    """y = W x  +  (alpha/r) * B (A x)

    W is the frozen pretrained matrix, kept in fp16 so it uses the T4 tensor cores.
    A and B are the trainable correction, kept in fp32 so their gradients do not
    underflow. The low-rank path is computed in fp32 and cast back on the way out.
    """
    def __init__(self, base: nn.Linear, r: int, alpha: float):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad_(False)          # the pretrained weight never moves
        self.r, self.alpha = r, alpha
        self.scaling = alpha / r
        dev = base.weight.device
        self.A = nn.Parameter(torch.empty(r, base.in_features, dtype=torch.float32, device=dev))
        self.B = nn.Parameter(torch.zeros(base.out_features, r, dtype=torch.float32, device=dev))
        self.reset()

    def reset(self):
        """Back to the identity map. This is how we discard a self-edit."""
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        nn.init.zeros_(self.B)               # B = 0  =>  delta_W = 0  =>  exactly the base model

    def forward(self, x):
        out = self.base(x)
        lora = (x.to(torch.float32) @ self.A.T) @ self.B.T
        return out + (lora * self.scaling).to(out.dtype)


def _is_lora(mod) -> bool:
    """Duck-typed check instead of isinstance.

    In a notebook, re-running the cell that defines LoRALinear creates a BRAND NEW
    class object. Adapters attached before that point are instances of the old class,
    so `isinstance(child, LoRALinear)` returns False and remove_lora() silently
    removes nothing while reporting success. That bug cost us a confusing run.
    Matching on structure instead of identity makes these functions re-run safe.
    """
    return all(hasattr(mod, a) for a in ("A", "B", "base", "scaling"))


# Paper, Appendix B.2: LoRA "applied to all MLP and attention projection layers".
# Appendix B.3 Table 3 gives the single-passage search space with rank 32 / alpha 64
# in bold as the selected values. We use the bolded single-passage values.
TARGETS = ("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj")

def inject_lora(m, r=32, alpha=64, targets=TARGETS):
    """Replace every targeted nn.Linear with a LoRALinear wrapping it. Returns handles.

    NOTE the first loop. Freezing only the wrapped Linear is not enough: embeddings,
    layernorms and lm_head keep requires_grad=True and torch allocates gradient
    buffers for all ~150M of them on every backward pass, even though the optimizer
    never sees them. On a 16 GB card that is the difference between fitting and not.
    A unit test below is what caught this.
    """
    remove_lora(m)                     # never stack an adapter on an adapter
    for p in m.parameters():
        p.requires_grad_(False)
    handles = []
    for _, module in list(m.named_modules()):
        for child_name, child in list(module.named_children()):
            if child_name in targets and isinstance(child, nn.Linear):
                wrapped = LoRALinear(child, r, alpha)
                setattr(module, child_name, wrapped)
                handles.append(wrapped)
    return handles

def remove_lora(m):
    """Put the plain nn.Linear back. The model returns to being exactly the base model."""
    n = 0
    for _, module in list(m.named_modules()):
        for child_name, child in list(module.named_children()):
            if _is_lora(child):
                setattr(module, child_name, child.base)
                n += 1
    return n

def reset_lora(handles):
    for h in handles:
        h.reset()

def lora_params(handles):
    return [p for h in handles for p in (h.A, h.B)]

print("LoRALinear, inject_lora, remove_lora, reset_lora defined")
print(f"stale adapters cleaned from model: {remove_lora(model)}")
print(f"model parameters now: {sum(p.numel() for p in model.parameters()):,}")

LoRALinear, inject_lora, remove_lora, reset_lora defined
stale adapters cleaned from model: 168
model parameters now: 494,032,768


In [17]:
remove_lora(model)                      # make this cell safe to re-run
probe = tok("The Scottish Parliament was established in", return_tensors="pt").to(DEV)

with torch.no_grad():
    base_logits = model(**probe).logits.float().clone()

handles = inject_lora(model, r=32, alpha=64)
n_lora = sum(p.numel() for p in lora_params(handles))
n_base = sum(p.numel() for p in model.parameters()) - n_lora

print(f"adapters injected      {len(handles)}  ({len(handles)//model.config.num_hidden_layers} per layer x {model.config.num_hidden_layers} layers)")
print(f"trainable parameters   {n_lora:,}  ({100*n_lora/n_base:.2f}% of the frozen model)")
print(f"frozen parameters      {n_base:,}")

# TEST 1 -- the rule that makes attach/detach safe.
# B is initialised to zero, so delta_W = B A = 0 and the adapted model must be
# bit-identical to the base model. Not approximately. Identical.
with torch.no_grad():
    lora_logits = model(**probe).logits.float()
max_dev = (lora_logits - base_logits).abs().max().item()
assert max_dev == 0.0, f"fresh adapter is not a no-op, max deviation {max_dev}"
print(f"\n[PASS] test 1: fresh adapter is exactly the identity (max deviation {max_dev})")

# TEST 2 -- the forward formula is literally y = Wx + (alpha/r) B A x.
h = handles[0]
with torch.no_grad():
    h.A.normal_(0, 0.02)
    h.B.normal_(0, 0.02)          # make B non-zero so there is something to check
    x = torch.randn(4, h.base.in_features, device=DEV, dtype=torch.float16)
    got = h(x)
    want = h.base(x) + ((x.float() @ h.A.T @ h.B.T) * (h.alpha / h.r)).to(torch.float16)
assert torch.allclose(got, want, atol=1e-3), "forward does not match the equation"
print(f"[PASS] test 2: forward matches y = Wx + (alpha/r)*B*A*x   (scaling = {h.scaling})")

# TEST 3 -- only the adapters can move.
# This test failed the first time it was run and exposed a real bug: see inject_lora.
trainable = [n for n, p in model.named_parameters() if p.requires_grad]
bad = [n for n in trainable if not n.endswith((".A", ".B"))]
assert not bad, f"something other than A/B is trainable: {bad[:3]}"
print(f"[PASS] test 3: exactly {len(trainable)} trainable tensors, all of them A or B")

# TEST 4 -- reset() really discards an edit, which is what the inner loop relies on.
reset_lora(handles)
with torch.no_grad():
    after_reset = model(**probe).logits.float()
assert (after_reset - base_logits).abs().max().item() == 0.0, "reset did not restore the base model"
print("[PASS] test 4: reset_lora() returns the model exactly to base")

# TEST 5 -- removal restores the original module objects.
n_removed = remove_lora(model)
with torch.no_grad():
    after_remove = model(**probe).logits.float()
assert (after_remove - base_logits).abs().max().item() == 0.0, "removal changed the model"
print(f"[PASS] test 5: remove_lora() detached {n_removed} adapters, model unchanged")
print("\nall structural rules hold. the adapter is safe to attach and throw away in a loop.")

adapters injected      168  (7 per layer x 24 layers)
trainable parameters   17,596,416  (3.56% of the frozen model)
frozen parameters      494,032,768

[PASS] test 1: fresh adapter is exactly the identity (max deviation 0.0)
[PASS] test 2: forward matches y = Wx + (alpha/r)*B*A*x   (scaling = 2.0)
[PASS] test 3: exactly 336 trainable tensors, all of them A or B
[PASS] test 4: reset_lora() returns the model exactly to base
[PASS] test 5: remove_lora() detached 168 adapters, model unchanged

all structural rules hold. the adapter is safe to attach and throw away in a loop.


---
# Part 4 — The inner loop

This is the machine the whole paper is built on. Its job:

```
text in  ->  finetune a fresh adapter on that text  ->  test the updated model  ->  score out  ->  throw the adapter away
```

In the paper's notation this is the step written `θ' ← SFT(θ, SE)` on line 5 of Algorithm 1. Everything else in SEAL is about deciding *what text to put in*.

Three things make it delicate.

**It has to be repeatable and independent.** Self-edit number 41 must be trained on a model that has never seen self-edits 1 through 40. If any state leaks between runs, the reward signal becomes a function of the order you happened to evaluate things in, and the RL loop will chase that instead of the task. Our `reset_lora()` guarantees this, and test 4 above is the proof.

**The training objective is ordinary next-token prediction.** There is nothing clever here. We take the self-edit text and run the standard causal language modelling loss over it. The paper: "we compute the standard causal language-modeling loss over each sequence `s_i`". The intelligence is entirely in *which sequences* get chosen, not in how they are trained on.

**One line of the self-edit is one training document.** The paper is specific about this in Appendix B.3: "In the single-passage case, we split it by newlines into a set of training documents." So a self-edit that lists eight implications becomes eight separate short sequences, not one long one. This matters more than it looks. Eight short independent documents give eight independent gradient signals and no cross-sentence context to lean on, which pushes the model to store each fact on its own rather than as a continuation of the previous sentence.

## Hyperparameters, from Appendix B.3 Table 3

The paper searched a grid and bolded what it selected for the single-passage setting. We take the bolded values:

| Parameter | Search space | Selected |
|---|---|---|
| LoRA rank `r` | [**32**, 64] | **32** |
| LoRA alpha `α` | [32, **64**] | **64** |
| Learning rate | [1e-4, 3e-4, 5e-4, **1e-3**, 2e-3] | **1e-3** |
| Epochs | [1, 5, **10**, 15, 20] | **10** |
| Batch size | [**1**, 4] | **1** |

A learning rate of 1e-3 for ten epochs on a handful of short sentences is an aggressive setting. It is meant to be. The point is to hammer a small amount of text into the weights hard enough that it survives the passage being taken away.


In [18]:
from dataclasses import dataclass

@dataclass
class SFTConfig:
    """Appendix B.3, Table 3, single-passage column (bolded values)."""
    lora_r: int = 32
    lora_alpha: int = 64
    lr: float = 1e-3
    epochs: int = 10
    batch_size: int = 1
    max_len: int = 256
    grad_clip: float = 1.0

CFG = SFTConfig()

def sft(handles, texts, cfg: SFTConfig = CFG, verbose=False):
    """Standard causal LM finetuning of ONLY the LoRA parameters, on a list of texts.

    Each element of `texts` is one independent training document (paper B.3: the
    self-edit is split on newlines). Returns the loss curve.
    """
    texts = [t for t in texts if t.strip()]
    if not texts:
        return []

    params = lora_params(handles)
    opt = torch.optim.AdamW(params, lr=cfg.lr, weight_decay=0.0)
    # fp16 gradients underflow to zero on small parameters. GradScaler multiplies the
    # loss by a large factor before backward and divides it out before the step, which
    # keeps the gradients inside fp16's representable range. Without this the adapter
    # trains to nothing and everything downstream silently reads "no effect".
    scaler = torch.amp.GradScaler("cuda")

    model.train()
    losses = []
    for ep in range(cfg.epochs):
        order = list(range(len(texts)))
        random.Random(ep).shuffle(order)
        for i in range(0, len(order), cfg.batch_size):
            batch = [texts[j] for j in order[i:i + cfg.batch_size]]
            enc = tok(batch, return_tensors="pt", padding=True, truncation=True,
                      max_length=cfg.max_len).to(DEV)
            labels = enc["input_ids"].clone()
            labels[enc["attention_mask"] == 0] = -100      # never train on padding
            out = model(**enc, labels=labels)
            opt.zero_grad(set_to_none=True)
            scaler.scale(out.loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, cfg.grad_clip)
            scaler.step(opt)
            scaler.update()
            losses.append(out.loss.item())
        if verbose:
            print(f"  epoch {ep+1:>2}  loss {np.mean(losses[-len(order):]):.4f}")
    model.eval()
    del opt, scaler
    return losses


def inner_loop(handles, train_texts, passage, cfg: SFTConfig = CFG):
    """The complete SEAL inner loop for one self-edit.

    1. discard whatever the adapter currently holds
    2. finetune on the supplied text
    3. answer the passage's questions WITHOUT the passage
    4. report accuracy under all three graders

    Returns the scores. The caller decides what to do with the adapter next.
    """
    reset_lora(handles)
    losses = sft(handles, train_texts, cfg)
    res = evaluate(model, [passage], with_context=False)
    return {"acc": res["acc"], "records": res["records"],
            "loss_first": losses[0] if losses else None,
            "loss_last": losses[-1] if losses else None,
            "n_docs": len([t for t in train_texts if t.strip()])}

print("inner loop defined")
print(f"config: {CFG}")

inner loop defined
config: SFTConfig(lora_r=32, lora_alpha=64, lr=0.001, epochs=10, batch_size=1, max_len=256, grad_clip=1.0)


In [19]:
# How long are the passages in tokens? If max_len truncates them, "train on the passage"
# would quietly become "train on the first half of the passage".
plens = [len(tok(p["context"])["input_ids"]) for p in EVAL_PASSAGES + TRAIN_PASSAGES]
print(f"passage length in tokens: min {min(plens)}  median {int(np.median(plens))}  max {max(plens)}")
CFG.max_len = int(max(plens)) + 16
print(f"max_len raised to {CFG.max_len} so no passage is ever truncated\n")

# ---- one inner loop, watched closely --------------------------------------------
p = EVAL_PASSAGES[0]
handles = inject_lora(model, CFG.lora_r, CFG.lora_alpha)

before = evaluate(model, [p], with_context=False)
print(f"passage: {p['title']}  ({len(p['qas'])} questions)")
print(f"before adaptation (fresh adapter = base model):  {before['acc']}")

t0 = time.time()
reset_lora(handles)
losses = sft(handles, [p["context"]], CFG, verbose=True)
t_train = time.time() - t0

after = evaluate(model, [p], with_context=False)
t_total = time.time() - t0

print(f"\nafter finetuning on the raw passage:             {after['acc']}")
print(f"\ntrain {t_train:.1f}s  +  eval {t_total-t_train:.1f}s  =  {t_total:.1f}s per inner loop")
print(f"loss {losses[0]:.3f} -> {losses[-1]:.3f}")
print(f"gpu peak {torch.cuda.max_memory_allocated(DEV)/2**30:.2f} GiB")

print("\nwhat the adapted model now says:")
for r in after["records"]:
    print(f"  Q {r['question']}")
    print(f"    gold {r['golds']}   pred {r['pred']!r}   {'HIT' if r[PRIMARY] else 'miss'}")

passage length in tokens: min 95  median 158  max 377
max_len raised to 393 so no passage is ever truncated

passage: Scottish_Parliament  (5 questions)
before adaptation (fresh adapter = base model):  {'EM': 0.0, 'Contains': 0.0, 'F1>=0.6': 0.0}
  epoch  1  loss 2.1448
  epoch  2  loss 1.2379
  epoch  3  loss 0.3802
  epoch  4  loss 0.3155
  epoch  5  loss 0.0701
  epoch  6  loss 0.0228
  epoch  7  loss 0.0411
  epoch  8  loss 0.0059
  epoch  9  loss 0.0246
  epoch 10  loss 0.0080

after finetuning on the raw passage:             {'EM': 0.0, 'Contains': 80.0, 'F1>=0.6': 0.0}

train 2.0s  +  eval 1.3s  =  3.4s per inner loop
loss 2.145 -> 0.008
gpu peak 2.19 GiB

what the adapted model now says:
  Q What day of the week are general elections held?
    gold ['Thursday']   pred 'The following general election is held on the first Thursday in May every four years (1999, 20'   HIT
  Q What month, every four years, are the ordinary general elections held on?
    gold ['May']   pred '1999 (i

### The second eval bug, and this one could have faked the entire paper

The demo above scored **80% under `Contains`** and **0% under both EM and F1**. Look at what the model actually produced:

> Q: What month, every four years, are the ordinary general elections held on?
> gold: `May`
> pred: `'1999 (i.e., 5 May 2011, 7 May 201'`

That is not an answer. It is a fragment of the passage, cut off mid-token, that happens to contain the string "May". `Contains` gave it full credit.

What happened is that finetuning for 10 epochs at lr 1e-3 on a single document drove the loss to 0.008. The model did not learn the passage, it **memorized** it, and now it responds to any question about that passage by emitting nearby passage text. The gold answer is a span *inside* the passage by the definition of SQuAD, so a model that regurgitates passage text will contain the gold answer very often, whether or not it has understood anything.

Now notice why this is dangerous rather than merely annoying. Earlier we measured the `Contains` grader's false-positive rate at 0.2%, and declared it safe. That measurement was taken on the **base** model, whose answers were 4.4 words long. It does not transfer. The grader's bias is a function of how verbose the model is, and finetuning makes the model much more verbose.

So the bias is **small before adaptation and large after adaptation**. SEAL is measured by exactly one comparison: accuracy before adaptation versus accuracy after adaptation. A grader whose generosity grows in that same direction will produce a large positive result from a model that learned nothing at all.

We must re-measure the control in the adapted state before we trust a single number from here on.


In [20]:
def run_per_passage(make_texts, passages, cfg=CFG, tag=""):
    """SEAL's single-passage protocol: one fresh adapter per passage, trained on that
    passage's text, evaluated on that passage's questions with the passage removed."""
    handles = inject_lora(model, cfg.lora_r, cfg.lora_alpha)
    recs, t0 = [], time.time()
    for i, p in enumerate(passages):
        out = inner_loop(handles, make_texts(p), p, cfg)
        recs += out["records"]
        if (i + 1) % 8 == 0:
            print(f"  {tag} {i+1}/{len(passages)}  {time.time()-t0:.0f}s")
    remove_lora(model)
    acc = {k: 100.0 * sum(r[k] for r in recs) / len(recs) for k in GRADERS}
    return {"acc": acc, "records": recs, "secs": time.time() - t0}

print("BASELINE A -- finetune on the raw passage only  (paper Table 2: 'Train on Passage')")
raw = run_per_passage(lambda p: [p["context"]], EVAL_PASSAGES, tag="raw")
print(f"  done in {raw['secs']:.0f}s\n")

# The control, re-measured IN THE ADAPTED STATE. This is the number that decides
# whether `Contains` is still a grader or has become a verbosity detector.
ctrl_base = shuffled_control(floor["records"])
ctrl_raw  = shuffled_control(raw["records"])

wl_base = np.mean([len(normalize(r["pred"]).split()) for r in floor["records"]])
wl_raw  = np.mean([len(normalize(r["pred"]).split()) for r in raw["records"]])

print("=" * 84)
print(f"{'grader':<10} {'base':>8} {'+raw passage':>14} {'gain':>8}  | {'random-gold: base':>19} {'adapted':>10}")
print("-" * 84)
for g in GRADERS:
    b, a = floor["acc"][g], raw["acc"][g]
    print(f"{g:<10} {b:>7.1f}% {a:>13.1f}% {a-b:>7.1f}  | {ctrl_base[g][0]:>18.1f}% {ctrl_raw[g][0]:>9.1f}%")
print("-" * 84)
print(f"{'pred words':<10} {wl_base:>7.1f}  {wl_raw:>13.1f}")
print("=" * 84)

print("\nInflation introduced by adaptation, per grader (credit for nothing):")
for g in GRADERS:
    d = ctrl_raw[g][0] - ctrl_base[g][0]
    share = 100 * d / max(raw["acc"][g] - floor["acc"][g], 1e-9)
    print(f"  {g:<10} control rose {d:>5.1f} points, which is {share:>6.1f}% of this grader's measured gain")
print(f"\npaper (Qwen2.5-7B): base 32.7 -> train on passage 33.5, a gain of +0.8")

BASELINE A -- finetune on the raw passage only  (paper Table 2: 'Train on Passage')
  raw 8/24  26s
  raw 16/24  51s
  raw 24/24  76s
  done in 76s

grader         base   +raw passage     gain  |   random-gold: base    adapted
------------------------------------------------------------------------------------
EM             0.7%           0.7%     0.0  |                0.0%       0.0%
Contains       3.6%          31.9%    28.3  |                0.3%       0.5%
F1>=0.6        2.9%           2.9%     0.0  |                0.0%       0.0%
------------------------------------------------------------------------------------
pred words     6.7           14.1

Inflation introduced by adaptation, per grader (credit for nothing):
  EM         control rose   0.0 points, which is    0.0% of this grader's measured gain
  Contains   control rose   0.1 points, which is    0.5% of this grader's measured gain
  F1>=0.6    control rose   0.0 points, which is    0.0% of this grader's measured gain

pap

### The verdict on that worry: raised, measured, and it did not hold

The control says the inflation from regurgitation is **0.1 points**, which is 0.5% of the measured gain. The five-question demo was a small-sample fluke, not a systematic bias. `Contains` survives and stays our primary grader.

That is worth stating plainly rather than quietly deleting. The concern was a real mechanism, the check was cheap, and it came back negative. A reproduction that only reports the worries that turned out to be true is not reporting honestly.

### What Baseline A actually found

| | base | + raw passage | gain | paper (7B) |
|---|---|---|---|---|
| EM | 0.7% | 0.7% | **0.0** | — |
| Contains | 3.6% | 31.9% | **+28.3** | 32.7 → 33.5, **+0.8** |
| F1 ≥ 0.6 | 2.9% | 2.9% | **0.0** | — |

The two graders disagree completely, and the disagreement *is* the result.

Under `Contains` the model gained 28 points. Under EM and F1 it gained exactly nothing. Both are true descriptions of the same behaviour: after memorizing the passage the model answers questions by **reciting nearby passage text**. The gold answer is a span inside the passage by SQuAD's construction, so recitation contains the answer, but recitation is not answering. EM and F1 demand that the model isolate the span, and it cannot.

So claim **C1** ("finetuning on the raw passage barely helps") comes out **confirmed in substance, refuted in surface form** at this scale. The paper's 7B model gains +0.8 because it already scores 32.7 and there is little headroom in recitation. Our 0.5B model starts near zero, so recitation buys a lot of `Contains` credit. But neither model learned to *answer*, which is the paper's actual point, and our EM column shows that with unusual clarity: **0.0 points gained**.

This is exactly the gap SEAL exists to close. The passage in its raw form is not a good teacher. The next part tests the paper's answer, which is to let the model rewrite the passage into its own notes first.
